## Pairing Terms

In [ ]:
# swapping terms: swap minority groups with majority groups

import re
import pandas as pd
identity_sample = pd.read_csv("/identity_samples.csv")

# Two way swap: Take comments of one group (eg. muslim), swap it with the other (eg. Christian). Take comments of the other (eg. Christian), swap it with the same group. (eg. muslim)
SWAPS = [
    #religion
    {"a": "muslim", "b": "christian", "two_way": True,
     "words": {"muslim": "christian", "muslims": "christians", "islam": "christianity"}},
    {"a": "jewish", "b": "christian", "two_way": True,
     "words": {"jewish": "christian", "jews": "christians", "judaism": "christianity"}},
    {"a": "atheist", "b": "christian", "two_way": True,
     "words": {"atheist": "christian", "atheists": "christians", "atheism": "christianity"}},
    {"a": "hindu", "b": "christian", "two_way": True,
     "words": {"hindu": "christian", "hindus": "christians", "hinduism": "christianity"}},
    {"a": "buddhist", "b": "christian", "two_way": True,
     "words": {"buddhist": "christian", "buddhists": "christians", "buddhism": "christianity"}},

    #race
    {"a": "black", "b": "white", "two_way": True,
     "words": {"black": "white", "blacks": "whites"}},
    {"a": "asian", "b": "white", "two_way": True,
     "words": {"asian": "white", "asians": "whites"}},
    {"a": "latino", "b": "white", "two_way": False,
     "words": {"latino": "white", "latinos": "whites",
               "hispanic": "white", "hispanics": "whites"}},
    #gender
    {"a": "female", "b": "male", "two_way": True,
     "words": {"woman": "man", "women": "men", "female": "male",
               "females": "males", "girl": "boy", "girls": "boys"}},
    {"a": "transgender", "b": "cisgender", "two_way": False,
     "words": {"transgender": "cisgender"}},
    {"a": "homosexual_gay_or_lesbian", "b": "heterosexual", "two_way": True,
     "words": {"gay": "straight", "gays": "straights",
               "homosexual": "heterosexual", "homosexuals": "heterosexuals"}},
    # Bisexual was dropped (only had 23 swappable comments)

    # {"a": "bisexual", "b": "heterosexual", "two_way": True,
    #  "words": {"bisexual": "straight", "bisexuals": "straights"}},

]
CONTROL = {"people": "neighbors"}




## Swap Functions

In [ ]:
# Phrases that are not about identity: protect them from swapping

VEHICLES = r"car|cars|truck|trucks|suv|suvs|van|vans|sedan|jeep|pick-?up|pickups|vehicle|lexus|ford|chevy|toyota|honda|bmw|mercedes|motorcycle"

OBJECTS = (r"shirt|shirts|t-shirt|hat|hats|dress|suit|suits|pants|jacket|coat|shoes|socks|gloves|robe|robes|hood|hoods|"
           r"sheet|sheets|helmet|helmets|smoke|cat|cats|dog|dogs|lab|horse|horses|bear|bears|paint|ink|tie|background|board|"
           r"sand|snow|beach|beaches|water|wine|bread|rice|sauce|tea|coffee|pepper|flag|flags|line|lines|screen|letters|text|"
           r"box|boxes|crayon|zone|fat|fence|picket|hair|beard|paper|papers|fur|feathers|stripe|stripes|tile|tiles|walls?")

# List of bad patterns, regex
BAD_PATTERNS = [
    # links and emails
    r"https?://\S+", r"\bwww\.\S+", r"\S+@\S+\.\w+",

    # ---- black / white: colors ----
    rf"\b(?:black|white)[\s-]+(?:{OBJECTS})\b",
    rf"\b(?:black|white)[\s-]+(?:[\w'-]+[\s-]+){{0,2}}(?:{VEHICLES})\b",
    r"\bblack[\s,-]+and[\s-]+white\b", r"\bwhite[\s,-]+and[\s-]+black\b",
    r"\bred,?[\s-]+white,?[\s-]+(?:and[\s-]+)?blue\b", r"\bblue[\s-]+and[\s-]+white\b",
    r"\bblack[\s-]+and[\s-]+(?:blue|gold|red)\b",
    r"\b(?:hair|beard)[\s\w']{0,25}?\b(?:turn|turns|turning|turned|went|gone|going)[\s-]+(?:white|black)\b",

    # ---- black / white: idioms, places, names ----
    r"\bwhite[\s-]+(?:house|houses|paper|papers|collar|lie|lies|noise|space|elephant|elephants|knight|knuckle|knuckles|"
    r"out|wash|washed|washing|wedding|christmas|easter|easters|meat|blood\s+cells?|sox|castle|helmets|flag|hat|hats|"
    r"board|boards|list|cliffs|nile|sea|russia|russian|russians|stripes|matter|magic)\b",
    r"\bgreat[\s-]+white[\s-]+(?:north|shark|sharks|hope)\b",
    r"\bblack[\s-]+(?:market|markets|hole|holes|eye|eyes|friday|sheep|ice|mold|belt|belts|list|listed|listing|out|outs|"
    r"widow|magic|gold|hawk|hawks|mail|mailed|mailing|op|ops|swan|swans|death|rock|sea|hills|forest|budget|box|boxes|"
    r"tie|humor|humour|comedy|mark|marks|plague|tuesday|monday|thursday|label|lung|mirror|ink|hat|hats|bear|bears)\b",
    r"\bin[\s-]+the[\s-]+black\b",
    r"\b(?:betty|walter|jack|barry|bill|mr\.?|mrs\.?|ms\.?|dr\.?)[\s-]+(?:white|black)\b",

    # ---- straight: not about orientation ----
    r"\bstraight[\s-]+(?:answer|answers|a's|as|a|line|lines|up|face|talk|talking|shooter|ticket|party|out|away|from|"
    r"to|into|through|jacket|forward|ahead|edge|razor|flush|shot|path|road|home|back|down|on|off|in|at|and\s+narrow)\b",
    r"\b(?:set|sets|setting|get|gets|got|getting|keep|keeps|kept|think|thinking|go|goes|went|going|stay|stays|stayed|"
    r"record|facts|story|stories|sit|stand|look|looked|looking|fly|flies)[\s-]+(?:[\w'-]+[\s-]+){0,2}straight\b",
    r"\b(?:\d+(?:st|nd|rd|th)?|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|several|second|third|fourth|fifth)[\s-]+straight\b",
    r"\bstraight[\s-]+(?:years?|days?|weeks?|months?|games?|seasons?|nights?|wins?|losses|quarters?|terms?|times|elections?|hours?)\b",

    # ---- man / men / boy / girl: sayings ----
    r"\boh[\s,]+(?:man|boy|girl)\b", r"\bboy[\s,]+oh[\s,]+boy\b", r"\b(?:atta|that'?s\s+my)[\s-]*(?:boy|girl)\b",
    r"\bman[\s-]+up\b", r"\bman[\s-]+made\b", r"\bman[\s-]+hours?\b", r"\bman[\s-]+to[\s-]+man\b",
    r"\bman'?s[\s-]+best[\s-]+friend\b", r"\bevery[\s-]+man[\s-]+for\b", r"\bno[\s-]+man'?s\b",
    r"\b(?:straw|middle|hit|best|front|common|family|wise|old|right[\s-]+hand|con|yes|lead|point|sand|snow|boogey|bogey)[\s-]+man\b",
    r"\b(?:boy|girl)[\s-]+scouts?\b", r"\bgirl[\s-]+friday\b", r"\bold[\s-]+boys?[\s-]+(?:club|network)\b",
    r"\bman[\s-]+of[\s-]+(?:the[\s-]+year|god|the[\s-]+people|his[\s-]+word|faith|science|letters)\b",

    # ---- religion: names and places ----
    r"\bchristian[\s-]+(?:bale|dior|grey|slater|louboutin|science\s+monitor)\b",
    r"\b(?:hans|jean|anne)[\s-]+christian\b",
    r"\bhindu[\s-]+kush\b",

    # ---- asian: the region, not people ----
    r"\basian[\s-]+(?:markets?|stocks?|carp|flu|countries|nations|economies|economy|region|regions|cuisine|food|"
    r"restaurant|restaurants|games|development\s+bank|infrastructure|tigers|allies|partners|neighbours|neighbors|"
    r"continent|currencies|trade|exports|imports|shares|demand|investors|leaders|summit)\b",
]

# joins all pattersn for search
BAD_RE = re.compile("|".join(f"(?:{p})" for p in BAD_PATTERNS), flags=re.IGNORECASE)

# finds every bad phrase and hides it
def protect(text):
    saved = []
    def hide(m):
        saved.append(m.group(0)) # Saves each real phrase in a list, so it can be put back later
        return f"\x00{len(saved)-1}\x00"
    return BAD_RE.sub(hide, text), saved

# Finds the codes and puts the saved phrases back in their place.
def unprotect(text, saved):
    return re.sub(r"\x00(\d+)\x00", lambda m: saved[int(m.group(1))], text)


# Test to see if masking function works: protected phrases on the left, real identity uses on the right
for t in ["He works in the White House.", "Putin is black mailing Trump.", "a white crew-cab Ford pick-up",
          "three straight years", "set the record straight", "oh man", "Asian markets fell",
          "white supremacists marched", "black people", "straight people", "gay men"]:
    print(f"{t:35} -> protected: {protect(t)[1]}")

He works in the White House.        -> protected: ['White House']
Putin is black mailing Trump.       -> protected: ['black mailing']
a white crew-cab Ford pick-up       -> protected: ['white crew-cab Ford pick-up']
three straight years                -> protected: ['three straight']
set the record straight             -> protected: ['set the record straight']
oh man                              -> protected: ['oh man']
Asian markets fell                  -> protected: ['Asian markets']
white supremacists marched          -> protected: []
black people                        -> protected: []
straight people                     -> protected: []
gay men                             -> protected: []


In [ ]:
# Match lowercase and uppercase when swapping (Ex: MUSLIM -> CHRISTIAN)
def match_case(old, new):
    if old.isupper():
        return new.upper()
    if old[0].isupper():
        return new.capitalize()
    return new

# swap identity word with the pair in word map
def swap_text(text, word_map):
  #build search with whole words: (muslim | muslims)
    pattern = r"\b(" + "|".join(re.escape(w) for w in word_map) + r")\b"
    def replace(m): # m = one match that was found
        old = m.group(0) #matched word
        return match_case(old, word_map[old.lower()]) # look up swap word in word_map, then match capitals
    return re.subn(pattern, replace, text, flags=re.IGNORECASE) # re.subn finds each match, makes m, calls replace(m)

#building swap set
rows = []
for s in SWAPS:
    a, b, words = s["a"], s["b"], s["words"]

    # inside one comment: flip both groups' words together
    swap_map = {**words, **{v: k for k, v in words.items()}}

    # which groups' comments to use
    directions = [(a, b), (b, a)] if s["two_way"] else [(a, b)]

    for source, target in directions:
        subset = identity_sample[identity_sample["group"] == source] #pick comments

        # go through each comment
        for _, r in subset.iterrows():
            masked, saved = protect(r["comment_text"]) # 1. hide bad phrases
            new_text, n = swap_text(masked, swap_map)  # 2. swap the rest
            new_text = unprotect(new_text, saved)  # 3. put bad phrases back

            if n > 0: #keep comments where word changed
                rows.append({"id": r["id"], "group": source,
                             "pair": f"{a}-{b}",
                             "direction": f"{source}->{target}",
                             "original_text": r["comment_text"],
                             "swapped_text": new_text, "n_swaps": n})

# control: neutral swap on comments with no group
for _, r in identity_sample[identity_sample["group"] == "none"].iterrows():
    new_text, n = swap_text(r["comment_text"], CONTROL)
    if n > 0:
        rows.append({"id": r["id"], "group": "none", "pair": "control",
                     "direction": "people->neighbors",
                     "original_text": r["comment_text"],
                     "swapped_text": new_text, "n_swaps": n})

swap_set = pd.DataFrame(rows)
print(swap_set["direction"].value_counts())

direction
white->black                               2922
white->asian                               2922
black->white                               2838
female->male                               2817
jewish->christian                          2784
male->female                               2731
muslim->christian                          2713
homosexual_gay_or_lesbian->heterosexual    2506
asian->white                               1156
christian->muslim                           942
christian->atheist                          942
christian->jewish                           941
christian->buddhist                         940
christian->hindu                            939
atheist->christian                          843
transgender->cisgender                      831
latino->white                               494
people->neighbors                           378
heterosexual->homosexual_gay_or_lesbian     145
hindu->christian                            118
buddhist->christian           

## Checking weird data and bad phrases

In [ ]:
# Shows original comments paired with swapped term
def show(direction, n=1000):
    sub = swap_set[swap_set["direction"] == direction]
    for i, r in sub.sample(min(n, len(sub)), random_state=1).iterrows():
        print(i, "|", r["original_text"][:500])
        print("   ->", r["swapped_text"][:500], "\n")

show("white->black")


14911 | Not everyone, is a Nazi.

Not all people, who wanted to preserve a Confederate statue, is a white supremacist or Nazi.

The American Civil War, gave those Confederate soldiers, the right to be remembered, for what they were fighting & dying for.
   -> Not everyone, is a Nazi.

Not all people, who wanted to preserve a Confederate statue, is a black supremacist or Nazi.

The American Civil War, gave those Confederate soldiers, the right to be remembered, for what they were fighting & dying for. 

15941 | Even worse.  An old white guy hating Talent Aquisition Specialist II perhaps?
   -> Even worse.  An old black guy hating Talent Aquisition Specialist II perhaps? 

14334 | I read this morning that a Trump administration might give us Attorney General Giuliani and Secretary of State Gingrich.  That makes my blood run cold!  Add to that the as yet unknown role of Steve Bannon -- with his roots in Breitbart, white supremacy and anti-Semitism -- and it seems difficult not to worry ab

In [ ]:
show("black->white")

12231 | So that means nobomba is a black monkey. Thanks for the comparison
   -> So that means nobomba is a white monkey. Thanks for the comparison 

12563 | Hugh -

You are criticizing slavery - the most severe racism.  We can all agree with that - especially as it applies to the history of much of the nation. 

But what part of  the kidnapping of blacks, the inhuman conditions afforded them, the murder carried out against them or their use as "animals of pleasure" was actually carried out against them here in Oregon - by the generation that created the University of Oregon and paid taxes and donated money and livestock to the building of Deady Hall
   -> Hugh -

You are criticizing slavery - the most severe racism.  We can all agree with that - especially as it applies to the history of much of the nation. 

But what part of  the kidnapping of whites, the inhuman conditions afforded them, the murder carried out against them or their use as "animals of pleasure" was actually carried o

In [ ]:
show("heterosexual->homosexual_gay_or_lesbian")

30533 | So the day you were born, you "decided" to be heterosexual -- sure you did. ROFL.
   -> So the day you were born, you "decided" to be homosexual -- sure you did. ROFL. 

30559 | Have you written here anything positive about such relationships? No. Therefore you have deliberately misrepresented them as (in your own, borrowed words) 'gravely sinful'. What a hateful thing to do! That's homophobia. 

These relationships, as with their heterosexual counterparts, can express heroic virtue, and can even be the ground of nascent holiness. But being a homophobe, you wouldn't, recognise this, would you, much less publicly admit it? Like Scripture, and the CCC.

Hadn't heard of Mr
   -> Have you written here anything positive about such relationships? No. Therefore you have deliberately misrepresented them as (in your own, borrowed words) 'gravely sinful'. What a hateful thing to do! That's homophobia. 

These relationships, as with their homosexual counterparts, can express heroic virtue

I realized from checking som swapped comments that there are bad phrases where racial identity terms are not racially charged. For example, "white house." After this cell i went back and implemented a masking step in the code above.

In [ ]:
# Check if bad phrases got removed successfully
print(swap_set["direction"].value_counts())
print("Black House left:", swap_set["swapped_text"].str.contains("Black House").sum())   # should be 0
swap_set["n_protected"] = swap_set["original_text"].apply(lambda t: len(BAD_RE.findall(t)))
print("swaps with a protected phrase:", (swap_set["n_protected"] > 0).sum())

direction
white->black                               2922
white->asian                               2922
black->white                               2838
female->male                               2817
jewish->christian                          2784
male->female                               2731
muslim->christian                          2713
homosexual_gay_or_lesbian->heterosexual    2506
asian->white                               1156
christian->muslim                           942
christian->atheist                          942
christian->jewish                           941
christian->buddhist                         940
christian->hindu                            939
atheist->christian                          843
transgender->cisgender                      831
latino->white                               494
people->neighbors                           378
heterosexual->homosexual_gay_or_lesbian     145
hindu->christian                            118
buddhist->christian           

In [ ]:
# check that there are no leftover codes from regex masking
leftover = swap_set["swapped_text"].str.contains("\x00").sum()
print("leftover codes:", leftover)   # should be 0

# save data to file
swap_set.to_csv("swap_set.csv", index=False)
print("saved", len(swap_set), "rows")

from google.colab import files
files.download("swap_set.csv")

leftover codes: 0
saved 31016 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Identity Group: Foundation Detection Rate